In [9]:

import torch, torch.nn as nn, torch.nn.functional as F, numpy as np


In [24]:

stride, padding, kernel_size = 2, 1, 3

x = torch.tensor([[[[4., 2., 3.], [4., 5., 6.], [7., 8., 9.]]]])
m = nn.ConvTranspose2d(1, 1, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
with torch.no_grad():
    m.weight[:] = torch.tensor([[[[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]]]])

x2d = x[0, 0]; kernel = m.weight[0, 0]
H_out = (x2d.shape[0]-1)*stride - 2*padding + kernel_size
print(f'Expected output: {H_out}x{H_out}')


Expected output: 5x5


In [25]:
kernel

tensor([[ 1.,  0., -1.],
        [ 2.,  0., -2.],
        [ 1.,  0., -1.]], grad_fn=<SelectBackward0>)

In [26]:

# Step 1
H, W = x2d.shape
x_strided = torch.zeros(H+(H-1)*(stride-1), W+(W-1)*(stride-1))
x_strided[::stride, ::stride] = x2d
print(f'Step 1: {tuple(x2d.shape)} -> {tuple(x_strided.shape)}')
print(x_strided.int())


Step 1: (3, 3) -> (5, 5)
tensor([[4, 0, 2, 0, 3],
        [0, 0, 0, 0, 0],
        [4, 0, 5, 0, 6],
        [0, 0, 0, 0, 0],
        [7, 0, 8, 0, 9]], dtype=torch.int32)


In [27]:

# Step 2
pad = kernel_size - 1 - padding
x_padded = F.pad(x_strided.unsqueeze(0).unsqueeze(0), [pad]*4).squeeze()
print(f'Step 2: {tuple(x_strided.shape)} -> {tuple(x_padded.shape)}')
print(x_padded.int())


Step 2: (5, 5) -> (7, 7)
tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 4, 0, 2, 0, 3, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 4, 0, 5, 0, 6, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 7, 0, 8, 0, 9, 0],
        [0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)


In [28]:

# Step 3
kernel_flipped = torch.flip(kernel, [0, 1])
kernel_flipped


tensor([[-1.,  0.,  1.],
        [-2.,  0.,  2.],
        [-1.,  0.,  1.]], grad_fn=<FlipBackward0>)

In [29]:
out_manual = F.conv2d(x_padded.unsqueeze(0).unsqueeze(0), kernel_flipped.unsqueeze(0).unsqueeze(0))
out_manual


tensor([[[[ 0., -4.,  0.,  2.,  0.],
          [ 0., -1.,  0.,  2.,  0.],
          [ 0.,  2.,  0.,  2.,  0.],
          [ 0.,  2.,  0.,  2.,  0.],
          [ 0.,  2.,  0.,  2.,  0.]]]], grad_fn=<ConvolutionBackward0>)

In [33]:
with torch.no_grad():
    out_pytorch = m(x)
print(f'Step 3 output shape: {tuple(out_manual.squeeze().shape)}')
print(f'Match: {torch.allclose(out_manual, out_pytorch)}')
print('Output:')
print(out_manual.squeeze().detach())


Step 3 output shape: (5, 5)
Match: True
Output:
tensor([[ 0., -4.,  0.,  2.,  0.],
        [ 0., -1.,  0.,  2.,  0.],
        [ 0.,  2.,  0.,  2.,  0.],
        [ 0.,  2.,  0.,  2.,  0.],
        [ 0.,  2.,  0.,  2.,  0.]])
